In [1]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# Prep

In [2]:
#hedge_funds = pd.read_csv('sftds_hedgefunds.csv')

In [3]:
#hf_unique = tuple(hedge_funds['0'].unique())

In [4]:
#isin = pd.read_excel('EA_ISINs.xlsx')

In [5]:
#unique_isin = tuple(isin['ISIN'])

# Monetary policy shocks

In [6]:
df = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\bond_timeseries_v2.csv')

In [7]:
df['ISIN'].str[:2].unique()

array(['DE', 'ES', 'FR', 'IT', 'AT', 'BE', 'FI', 'GR', 'IE', 'LU', 'NL',
       'PT', 'SI', 'SK'], dtype=object)

In [8]:
df['Dates'] = pd.to_datetime(df['Dates'])
df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30948\996206371.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])


In [9]:
# calculate residual maturity in years
df['residual_bond_maturity'] = ((df['bond_maturity'] - df['Dates']).dt.days / 365)

In [10]:
df = df[df['residual_bond_maturity'] > 0]

In [11]:
df['amt_issued'] = (
    df['amt_issued']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace(' ', '', regex=False)
)

# convert to float, coercing invalid entries to NaN
df['amt_issued'] = pd.to_numeric(df['amt_issued'], errors='coerce')
df['amt_issued'] = df['amt_issued']/1e9

In [12]:
df['amt_issued'].fillna(df['amt_issued'].median(), inplace=True)

In [13]:
df['collateral_country'] = df['ISIN'].str[:2]

In [14]:
df = df[df['collateral_country'].isin(['DE','IT'])]

In [15]:
df = df[df['PX_ASK'].isna() == False]

In [16]:
securities = tuple(df['ISIN'].unique())

In [17]:
len(securities)

746

In [18]:
len(df)

484936

In [19]:
# Step 1: Identify the dealer list (counterparties to Cayman IFs)
query_dealers = f"""
SELECT DISTINCT dealer_id FROM (
    SELECT s.lender_id AS dealer_id
    FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
    LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
        ON s.borrower_id = s_borrower.id
    WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
      AND s.nominal_ccy = 'EUR'
      AND s.central_clearing = 'non-cleared'
      AND s.borrower_country_residence = 'KY' 
      AND s_borrower.sector = 'IF'
      AND s.security_isin IN {securities}
    UNION
    SELECT s.borrower_id AS dealer_id
    FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
    LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
        ON s.lender_id = s_lender.id
    WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
      AND s.nominal_ccy = 'EUR'
      AND s.central_clearing = 'non-cleared'
      AND s.lender_country_residence = 'KY' 
      AND s_lender.sector = 'IF'
      AND s.security_isin IN {securities}
  )
"""
df_dealers = pd.read_sql_query(query_dealers, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30948\1566539105.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dealers = pd.read_sql_query(query_dealers, cnxn)


In [20]:
dealer_ids = tuple(df_dealers['dealer_id'].unique())
print(f"Number of dealers facing HFs: {len(dealer_ids)}")

Number of dealers facing HFs: 18


In [21]:
pd.DataFrame(dealer_ids).to_excel('dealers.xlsx')

In [22]:
# Step 2a: HF borrowing cash from dealers (HF is borrower, dealer is lender)
# → HF posts bond to dealer → proxy for HF long position
query_hf_borrow = f"""
SELECT s.business_date,
       s.security_isin AS isin,
       s.lender_id AS dealer_id,
       SUM(s.nominal_euro) AS borrowing_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
  AND s.nominal_ccy = 'EUR'
  AND s.central_clearing = 'non-cleared'
  AND s.borrower_country_residence = 'KY'
  AND s_borrower.sector = 'IF'
  AND s.lender_id IN {dealer_ids}
  AND s.gnlcoll = 'SPEC'
  AND s.security_isin IN {securities}
  AND s.assttp_scty_issr_sector_riad = 'S1311'
GROUP BY s.business_date, s.security_isin, s.lender_id
"""
df_hf_borrow = pd.read_sql_query(query_hf_borrow, cnxn)

# Step 2b: HF lending cash to dealers (HF is lender, dealer is borrower)
# → HF receives bond from dealer → proxy for HF short position
query_hf_lend = f"""
SELECT s.business_date,
       s.security_isin AS isin,
       s.borrower_id AS dealer_id,
       SUM(s.nominal_euro) AS lending_volume
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender 
    ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
  AND s.nominal_ccy = 'EUR'
  AND s.central_clearing = 'non-cleared'
  AND s.lender_country_residence = 'KY'
  AND s_lender.sector = 'IF'
  AND s.borrower_id IN {dealer_ids}
  AND s.gnlcoll = 'SPEC'
  AND s.security_isin IN {securities}
  AND s.assttp_scty_issr_sector_riad = 'S1311'
GROUP BY s.business_date, s.security_isin, s.borrower_id
"""
df_hf_lend = pd.read_sql_query(query_hf_lend, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30948\1459948808.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hf_borrow = pd.read_sql_query(query_hf_borrow, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_30948\1459948808.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hf_lend = pd.read_sql_query(query_hf_lend, cnxn)


In [23]:
df_hf = df_hf_borrow.merge(df_hf_lend, on=['business_date','isin','dealer_id'], how='outer')

In [24]:
df_hf['borrowing_volume'] = df_hf['borrowing_volume'].fillna(0)
df_hf['lending_volume']   = df_hf['lending_volume'].fillna(0)
df_hf['net_pos'] = (df_hf['borrowing_volume'] - df_hf['lending_volume']) / 1e9

In [25]:
df_hf['business_date'] = pd.to_datetime(df_hf['business_date'])

In [28]:
# 1. Get the universe of dealers (from df_wide, since df has no dealer info)
dealers = df_hf['dealer_id'].unique()

# 2. Build the full (date, ISIN) panel from df — one row per bond per active day
date_isin = df[['Dates', 'ISIN']].drop_duplicates()

# 3. Cross-join with dealers to get every (date, ISIN, dealer) combination
date_isin['_key'] = 1
dealers_df = pd.DataFrame({'dealer_id': dealers, '_key': 1})
panel = date_isin.merge(dealers_df, on='_key').drop(columns='_key')

# 4. Left-join df_wide's intermediation values onto the panel
panel = panel.merge(
    df_hf,
    left_on=['Dates', 'ISIN', 'dealer_id'],
    right_on=['business_date', 'isin', 'dealer_id'],
    how='left'
)

# 5. Fill missing intermediation values with 0 (dealer was active but didn't trade this bond)
borrow_lend_cols = ['borrowing_volume', 
                    'lending_volume',
                    'net_pos']
panel[borrow_lend_cols] = panel[borrow_lend_cols].fillna(0)

# 6. Drop the duplicate key columns from df_wide
panel = panel.drop(columns=['business_date', 'isin'])

# 7. Join the bond attributes from df
result = panel.merge(df, on=['Dates', 'ISIN'], how='left')

In [29]:
query_bank_borrow = f"""
SELECT DISTINCT business_date
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower 
    ON s.borrower_id = s_borrower.id
WHERE s.business_date BETWEEN DATE '2021-01-04' AND DATE '2025-11-01'
  AND s.nominal_ccy = 'EUR'
  AND s.central_clearing = 'cleared'
  AND s.lender_id IN {dealer_ids}             -- dealer on other side
  AND s.gnlcoll = 'SPEC'
  AND s.security_isin IN {securities}
"""
df_dates= pd.read_sql_query(query_bank_borrow, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_30948\1036095019.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dates= pd.read_sql_query(query_bank_borrow, cnxn)


In [30]:
result = result[result['Dates'].isin(df_dates['business_date'].unique())]

In [31]:
shocks_all = pd.read_excel('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\altavilla.xlsx', sheet_name = 'Monetary Event Window')

In [32]:
shocks_all['date'] = pd.to_datetime(shocks_all['date'], dayfirst=True)

In [33]:
result.head()

,Dates,ISIN,dealer_id,borrowing_volume,lending_volume,net_pos,Unnamed: 0,PX_ASK,PX_BID,YLD_YTM_ASK,YLD_YTM_BID,bond_maturity,amt_issued,rating_issr,residual_bond_maturity,collateral_country
2718,2021-01-04,DE0001030203,549300FH0WJAPEHTIQ77,0.0,0.0,0.0,712,100.184,100.163,-0.675,-0.598,2021-04-14,18.0,Aaau,0.273973,DE
2719,2021-01-04,DE0001030203,2ZCNRR8UK83OBTEK2170,0.0,0.0,0.0,712,100.184,100.163,-0.675,-0.598,2021-04-14,18.0,Aaau,0.273973,DE
2720,2021-01-04,DE0001030203,5493006QMFDDMYWIAM13,0.0,0.0,0.0,712,100.184,100.163,-0.675,-0.598,2021-04-14,18.0,Aaau,0.273973,DE
2721,2021-01-04,DE0001030203,3TK20IVIUJ8J3ZU0QE75,0.0,0.0,0.0,712,100.184,100.163,-0.675,-0.598,2021-04-14,18.0,Aaau,0.273973,DE
2722,2021-01-04,DE0001030203,O2RNE8IBXP4R0TD8PU41,0.0,0.0,0.0,712,100.184,100.163,-0.675,-0.598,2021-04-14,18.0,Aaau,0.273973,DE


In [34]:
out = (
    result.merge(shocks_all[['date', 'OIS_2Y']], left_on=['Dates'], right_on=['date'], how='left')
       .drop(columns=['date'])  # helpers
)

In [35]:
out['OIS_2Y'].fillna(0, inplace=True)

In [36]:
out['bid_ask_spread'] = out['PX_ASK'] - out['PX_BID'] 

In [37]:
ctd = pd.read_excel('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\CTD.xlsx')

In [38]:
# month-end of (Year, Month)
ctd['period_end'] = pd.to_datetime(dict(year=ctd['Year'], month=ctd['Month'], day=1)) + pd.offsets.MonthEnd(0)
ctd['period_start'] = ctd['period_end'] - pd.DateOffset(years=1)

In [39]:
out = out.rename(columns={'ISIN': 'isin'})

In [40]:
out = out.rename(columns={'Dates': 'business_date'})

In [41]:
# --- expand & mark in-window ---
m = out.merge(ctd[['isin','period_start','period_end']], on='isin', how='left')
m['in_window'] = (m['business_date'] >= m['period_start']) & (m['business_date'] <= m['period_end'])

# collapse back to one row per (ISIN, business_date): 1 if any window matches
flag_df = (m.groupby(['isin','business_date'], as_index=False)['in_window']
             .any()
             .rename(columns={'in_window':'ctd_flag'}))

# --- attach flag to original `out` ---
out = out.merge(flag_df, on=['isin','business_date'], how='left')
out['ctd_flag'] = out['ctd_flag'].fillna(False).astype(int)


In [42]:
germany = pd.read_csv('NelsonSiegel_DE_prices.csv')

In [43]:
italy = pd.read_csv('NelsonSiegel_IT_prices.csv')

In [44]:
#spain = pd.read_csv('NelsonSiegel_ES_prices.csv')

In [45]:
#france = pd.read_csv('NelsonSiegel_FR_prices.csv')

In [46]:
df_duration = pd.concat([germany, italy], ignore_index=True)

In [47]:
df_duration['refdate'] = pd.to_datetime(df_duration['refdate'])

In [48]:
out = out.merge(df_duration[['refdate', 'bondcode', 'duration']], left_on = ['business_date', 'isin'], right_on = ['refdate', 'bondcode'], how = 'left')

In [49]:
out['duration'].fillna(out['residual_bond_maturity'], inplace = True)

In [50]:
len(out)

8496612

In [51]:
# One row per isin-date with the mid yield
yld_panel = (out[['isin','business_date','YLD_YTM_BID','YLD_YTM_ASK']]
             .drop_duplicates(['isin','business_date'])
             .sort_values(['isin','business_date'])
             .copy())
yld_panel['yld_mid'] = (yld_panel['YLD_YTM_BID'] + yld_panel['YLD_YTM_ASK']) / 2
yld_panel['delta_y'] = yld_panel.groupby('isin')['yld_mid'].diff() * 100  # bps

In [52]:
# Merge back onto the dealer-level panel
out = out.merge(
    yld_panel[['isin','business_date','yld_mid','delta_y']],
    on=['isin','business_date'], how='left'
)

In [53]:
out['delta_y'].describe()

count    8.439480e+06
mean     1.300921e-01
std      1.264804e+01
min     -5.538000e+03
25%     -2.350000e+00
50%      0.000000e+00
75%      2.850000e+00
max      7.167000e+02
Name: delta_y, dtype: float64

In [54]:
out = out[(out['delta_y'] <= 40) & (out['delta_y'] >= -40)]

In [55]:
# 1. Base Intensity: Rolling mean of |net_pos| (Magnitude only)
out['abs_net'] = (out['net_pos'].abs() / out['amt_issued']) * 100

In [56]:
out = out.sort_values(['dealer_id','isin','business_date']).reset_index(drop=True)

hf_roll = (
    out.groupby('isin')['abs_net']
     .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
     .reset_index(level=0, drop=True)   # <-- align index with d
)

out['hf_intensity_pre'] = hf_roll

In [57]:
# create binary
out.loc[(out['hf_intensity_pre'] > 0), 'hf_involved'] = 1
out['hf_involved'].fillna(0, inplace = True)

In [58]:
cds = pd.read_excel('dealer_cds.xlsx', sheet_name='Sheet2')

In [59]:
import pandas as pd

# 1. Build the column name -> LEI mapping
# Map from Bloomberg ticker in column name to LEI
ticker_to_lei = {
    'BACRED':      'PSNL19R2RXX5U3QWHI44',
    'HSBC':        'F0HUI1NY1AZMJMD8LP67',
    'DANBNK':      'MAES062Z21O4RZ2U7M96',
    'BOFA':        '549300FH0WJAPEHTIQ77',
    'BANCO BILBAO':'K8MS7FD7N5Z2WQ51AZ71',  # BBVASM in your mapping
    'SOCGEN':      'O2RNE8IBXP4R0TD8PU41',
    'SEB':         'F3JS33DEI6XQ4ZBPTN86',
    'BARCLAY':     '2G5BKIC2CB69PRJH1W31',  # BACR in your mapping
    'BNP':         'R0MUWSFPU8MPRO8K5P83',
    'HVB':         '2ZCNRR8UK83OBTEK2170',
    'CITDEL':      '549300G3XW84WZXTDL39',  # Citadel
    'KNFP':        'KX1WK48MPD4Y2NCUIZ63',
    'KBC BK':      '6B2PBRV1FCJDMR45RZ53',
    'DB':          '7LTWFZYICNSX8D621K86',
    'ACAFP A':     '1VUV7VQFKUOQSJ21A208',
    'CMZB':        '851WYGNLUQLFZBSYGB56',
    'SANTA':       '5493006QMFDDMYWIAM13',  # SANTAN
    'INGHOLDCO':   '3TK20IVIUJ8J3ZU0QE75',  # INTNED
}

# 2. Match each column to a ticker and rename to LEI
def match_ticker(col_name):
    for ticker in ticker_to_lei:
        if col_name.startswith(ticker):
            return ticker_to_lei[ticker]
    return None

# Apply renaming
new_cols = {col: match_ticker(col) for col in cds.columns if col != 'Dates'}
# Verify no unmatched columns
unmatched = [col for col, lei in new_cols.items() if lei is None]
if unmatched:
    print("Unmatched columns:", unmatched)

cds_renamed = cds.rename(columns=new_cols)

# 3. Melt wide to long
cds_long = cds_renamed.melt(
    id_vars='Dates',
    var_name='lei',
    value_name='cds'
)

# 4. Clean up
cds_long = cds_long.rename(columns={'Dates': 'business_date'})
cds_long['business_date'] = pd.to_datetime(cds_long['business_date'])
cds_long = cds_long.dropna(subset=['cds'])
cds_long = cds_long.sort_values(['business_date', 'lei']).reset_index(drop=True)

In [60]:
cds_long.head()

,business_date,lei,cds
0,2021-01-01,1VUV7VQFKUOQSJ21A208,52.710
1,2021-01-01,2G5BKIC2CB69PRJH1W31,57.635
2,2021-01-01,2ZCNRR8UK83OBTEK2170,37.743
3,2021-01-01,3TK20IVIUJ8J3ZU0QE75,39.385
4,2021-01-01,5493006QMFDDMYWIAM13,35.055


In [61]:
len(out)

8422704

In [62]:
# Before merging to out, compute rolling change in cds_long
cds_long = cds_long.sort_values(['lei', 'business_date'])
cds_long['cds_lag'] = cds_long.groupby('lei')['cds'].shift(1)
cds_long['cds_change_20d'] = cds_long.groupby('lei')['cds_lag'].diff(20)  # 20-day change
cds_long['cds_change_5d'] = cds_long.groupby('lei')['cds_lag'].diff(5)    # 5-day change

# Merge into out
out = out.drop(columns=['cds', 'lei', 'cds_lag'], errors='ignore')
out = out.merge(
    cds_long[['business_date', 'lei', 'cds', 'cds_lag', 'cds_change_20d', 'cds_change_5d']],
    left_on=['business_date', 'dealer_id'],
    right_on=['business_date', 'lei'],
    how='left'
)

In [65]:
out.to_csv('monetary_policy_induced_position_dealers.csv')